# 01 — Stage 2 contact sheet
한 face × 한 species 별로 ip_scale을 한 행으로 펼쳐 본다. seed=0 고정.

In [ ]:
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
runs = sorted(Path('experiments/runs').glob('stage2_*'))
RUN = runs[-1]
print('run:', RUN)
df = pd.read_parquet(RUN / 'metadata.parquet')
df = df[df['image_path'] != ''].copy()
print(df['face_id'].unique(), df['species'].unique(), sorted(df['ip_scale'].unique()))

In [ ]:
SEED = 0
scales = sorted(df['ip_scale'].unique())
faces  = sorted(df['face_id'].unique())
species_list = sorted(df['species'].unique())

for fid in faces:
    sub = df[(df['face_id'] == fid) & (df['seed'] == SEED)]
    if sub.empty: continue
    face_img = Image.open(sub.iloc[0]['face_path'])
    n_cols = 1 + len(scales)
    fig, axes = plt.subplots(len(species_list), n_cols, figsize=(2.4*n_cols, 2.4*len(species_list)))
    if len(species_list) == 1: axes = [axes]
    for r, sp in enumerate(species_list):
        axes[r][0].imshow(face_img); axes[r][0].set_title('input' if r == 0 else ''); axes[r][0].set_ylabel(sp); axes[r][0].set_xticks([]); axes[r][0].set_yticks([])
        for c, s in enumerate(scales):
            row = sub[(sub['species'] == sp) & (sub['ip_scale'] == s)]
            axes[r][c+1].axis('off')
            if not row.empty:
                axes[r][c+1].imshow(Image.open(row.iloc[0]['image_path']))
                if r == 0: axes[r][c+1].set_title(f'ip={s:.1f}')
    fig.suptitle(f'face: {fid}'); plt.tight_layout(); plt.show()